# M3 candidate subset

We build the training subset every Milestone 3 run trains on: 15% of each class's training photos, at least 20, with photographers taking turns, at seed 42. The notebook checks that a subset of about 16% still represents the full training split, and it also sets aside the calibration set and a 200-photo hand-check list.

**Inputs:** the dataset `serpentguard-indian-snakes-clean`, found with `kagglehub.dataset_download`.

**Outputs** in `/kaggle/working/`: `subset.csv`, `subset_report.json`, `calibration.csv`, `hand_check.csv` and `hand_check_sheet.jpg`. `subset.csv` and `subset_report.json` become the private dataset `serpentguard-m3-subset`.

The code that draws the subset and checks it is in this notebook; the calibration set and the subset's fraction, floor and seed come from `serpentguard.subset`, which the runs also read.

**Run:** on Kaggle, CPU only, with no parameters.

In [1]:
import json
import sys
from pathlib import Path

import kagglehub

CODE = Path("/kaggle/input/serpentguard-code/src")
sys.path.insert(0, str(CODE if CODE.exists() else Path("../src").resolve()))

import pandas as pd

OWNER = "puneetbajajiitm"


def input_root(slug):
    """Where an attached dataset is mounted; Kaggle no longer mounts datasets at /kaggle/input/<slug>."""
    return Path(kagglehub.dataset_download(f"{OWNER}/{slug}"))


DATA = input_root("serpentguard-indian-snakes-clean")
OUT = Path("/kaggle/working")
SUBSET_COLUMNS = ["path", "species", "observer_id", "split"]

In [2]:
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

from serpentguard.subset import NOT_LISTED, SEED, SUBSET_FLOOR, SUBSET_FRACTION, calibration_set

HAND_CHECK_PHOTOS, STRATUM_FLOOR = 200, 10
THUMB_SIZE, SHEET_COLUMNS, LABEL_HEIGHT = 160, 10, 24
ORIENTATION_TOLERANCE = 1.02  # M2: portrait means height > 1.02 x width


def make_subset(train, fraction=SUBSET_FRACTION, floor=SUBSET_FLOOR, seed=SEED):
    """Per class: fraction of its photos, at least floor (all if fewer); photographers take turns."""
    kept = []
    for _, group in train.groupby("species"):
        size = min(len(group), max(floor, int(round(fraction * len(group)))))
        shuffled = group.sample(frac=1, random_state=seed)
        shuffled = shuffled.assign(turn=shuffled.groupby("observer_id").cumcount())
        kept.append(shuffled.sort_values("turn", kind="stable").head(size).drop(columns="turn"))
    return pd.concat(kept)


def js_divergence(p, q):
    """Jensen-Shannon divergence in nats between two class-share vectors."""
    mean = (p + q) / 2

    def kl(a, b):
        nonzero = a > 0
        return float((a[nonzero] * np.log(a[nonzero] / b[nonzero])).sum())

    return (kl(p, mean) + kl(q, mean)) / 2


def class_mix(frame, classes):
    counts = frame["species"].value_counts().reindex(classes, fill_value=0).to_numpy(dtype=float)
    return counts / counts.sum()


def subset_report(full, subset):
    classes = sorted(full["species"].unique())
    calibration = calibration_set(full, subset)
    return {
        "photos": len(subset),
        "share": len(subset) / len(full),
        "classes": int(subset["species"].nunique()),
        "photographers": int(subset["observer_id"].nunique()),
        "india": float(subset["in_india"].mean()),
        "who_listed": float((subset["who_category"] != NOT_LISTED).mean()),
        "portrait": float((subset["height"] > ORIENTATION_TOLERANCE * subset["width"]).mean()),
        "js_divergence": js_divergence(class_mix(full, classes), class_mix(subset, classes)),
        "per_tier": {tier: {"full": float((full["tier"] == tier).mean()), "subset": float((subset["tier"] == tier).mean())}
                     for tier in sorted(full["tier"].unique())},
        "calibration": {"photos": len(calibration), "classes": int(calibration["species"].nunique())},
    }


def stratum_quotas(sizes, n):
    """STRATUM_FLOOR photos per stratum (all if fewer), the rest in proportion to size by largest remainder."""
    quotas = pd.Series(min(STRATUM_FLOOR, n // len(sizes)), index=sizes.index).clip(upper=sizes)
    spare = n - int(quotas.sum())
    while spare > 0 and (sizes - quotas).sum() > 0:
        room = sizes - quotas
        ideal = room * min(spare, room.sum()) / room.sum()
        add = np.floor(ideal).astype(int)
        leftover = min(spare, int(room.sum())) - int(add.sum())
        add[(ideal - add).sort_values(ascending=False, kind="stable").index[:leftover]] += 1
        quotas += add
        spare = n - int(quotas.sum())
    return quotas


def hand_check_sample(train, n=HAND_CHECK_PHOTOS, seed=SEED):
    """n training photos stratified by tier and WHO category."""
    strata = ["tier", "who_category"]
    sizes = train.groupby(strata).size()
    quotas = stratum_quotas(sizes, n)
    picked = [train.merge(pd.DataFrame([key], columns=strata), on=strata).sample(n=int(quotas[key]), random_state=seed)
              for key in sizes.index]
    return pd.concat(picked).reset_index(drop=True)


def contact_sheet(frame, image_root, out_path, thumb=THUMB_SIZE, cols=SHEET_COLUMNS):
    """JPEG grid of thumbnails with species and photo id under each."""
    cell = thumb + LABEL_HEIGHT
    sheet = Image.new("RGB", (cols * thumb, -(-len(frame) // cols) * cell), (255, 255, 255))
    draw = ImageDraw.Draw(sheet)
    for index, record in enumerate(frame.itertuples()):
        left, top = (index % cols) * thumb, (index // cols) * cell
        try:
            with Image.open(f"{image_root}/{record.path}") as image:
                tile = image.convert("RGB")
                tile.thumbnail((thumb, thumb))
        except OSError:
            tile = Image.new("RGB", (thumb, thumb), (90, 90, 90))
        sheet.paste(tile, (left + (thumb - tile.width) // 2, top))
        draw.text((left + 2, top + thumb + 1), f"{record.species}"[:28], fill=(0, 0, 0))
        draw.text((left + 2, top + thumb + 11), f"{record.photo_id}", fill=(0, 0, 0))
    sheet.save(out_path, quality=85)

In [3]:
metadata = pd.read_csv(DATA / "metadata.csv", dtype={"observer_id": str})
train = metadata[metadata["split"] == "train"]
subset = make_subset(train)
calibration = calibration_set(train, subset)
report = subset_report(train, subset)
subset[SUBSET_COLUMNS].to_csv(OUT / "subset.csv", index=False)
calibration[SUBSET_COLUMNS].to_csv(OUT / "calibration.csv", index=False)
(OUT / "subset_report.json").write_text(json.dumps(report, indent=2))
report

{'photos': 13925,
 'share': 0.16369449962970364,
 'classes': 184,
 'photographers': 6576,
 'india': 0.3972710951526032,
 'who_listed': 0.12114901256732495,
 'portrait': 0.40172351885098745,
 'js_divergence': 0.012217651955429277,
 'per_tier': {'tier1': {'full': 0.8733233803942775,
   'subset': 0.8005026929982046},
  'tier2': {'full': 0.12667661960572255, 'subset': 0.19949730700179533}},
 'calibration': {'photos': 7999, 'classes': 111}}

In [4]:
hand_check = hand_check_sample(train)
hand_check.to_csv(OUT / "hand_check.csv", index=False)
contact_sheet(hand_check, DATA / "data", OUT / "hand_check_sheet.jpg")

In [5]:
sample = subset.sample(20, random_state=42)
missing = [path for path in sample["path"] if not (DATA / "data" / path).exists()]
assert not missing, missing
assert not (DATA / "data" / "train/not_a_species/made_up.jpg").exists()  # negative control
print("20 of 20 sampled subset paths exist; the made-up path does not")

20 of 20 sampled subset paths exist; the made-up path does not
